In [2]:
import sys
from pathlib import Path

project_root = Path.cwd().parents[2]
print(project_root)
sys.path.append(str(project_root))

/Users/yopchi/Firenodes/Projects/FIREgen


In [3]:
from IPython.display import Image, display
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import START, StateGraph

from my_app.chatbot.chat_core.nodes import (
    analyze_user_answers,
    call_llm,
    compact_user_answer,
    create_followup_qa,
    determine_next_node,
    generate_greeting_message,
    present_predefined_questions,
    process_human_input_tool,
    summarize_user_profile,
    update_user_profile,
)
from my_app.chatbot.chat_core.state import InputState, OutputState, OverallState

workflow = StateGraph(OverallState, input_schema=InputState, output_schema=OutputState)
workflow.add_node("start_chat", generate_greeting_message)
workflow.add_node("ask_to_new_user", present_predefined_questions)
workflow.add_node("agent", call_llm)
workflow.add_node("add_qa", create_followup_qa)
workflow.add_node("compact_user_answer", compact_user_answer)
workflow.add_node("determine_next", determine_next_node)
workflow.add_node("request_input", process_human_input_tool)
workflow.add_node("analyze_answers", analyze_user_answers)
workflow.add_node("update_user_profile", update_user_profile)
workflow.add_node("summarize_user_profile", summarize_user_profile)

memory = InMemorySaver()

workflow.add_edge(START, "start_chat")
workflow.add_edge("start_chat", "ask_to_new_user")
# workflow.add_edge("ask_to_new_user", "agent")

graph = workflow.compile(checkpointer=memory)

display(Image(graph.get_graph().draw_mermaid_png(max_retries=5, retry_delay=2.0)))


ValueError: Failed to reach https://mermaid.ink/ API while trying to render your graph after 5 retries. To resolve this issue:
1. Check your internet connection and try again
2. Try with higher retry settings: `draw_mermaid_png(..., max_retries=5, retry_delay=2.0)`
3. Use the Pyppeteer rendering method which will render your graph locally in a browser: `draw_mermaid_png(..., draw_method=MermaidDrawMethod.PYPPETEER)`

In [ ]:
from my_app.chatbot.chat_core.state import InputState
from my_app.chatbot.services import MockProfileService

user_data = InputState(
    target_profile_category=[
        "investment_goal",
        "investment_emotions",
        "interests_categories",
        "investment_level",
        "knowledge_level",
        "risk_tolerance",
    ],
    user_meta_data={
        "name": "강요셉",
        "profile_status": "onboarding",
        "investment_goal": [],
        "investment_emotions": [],
        "interests_categories": [],
        "investment_level": [],
        "knowledge_level": [],
        "risk_tolerance": 0,
    },
)

config = {"configurable": {"thread_id": "1", "profile_service": MockProfileService()}}

message_data = []
events = []

for event in graph.stream(user_data, config, stream_mode="values"):
    events.append(event)
    if "messages" in event:
        data = event["messages"]
        message_data.append(data)
        if isinstance(data, list):
            for message in data:
                message.pretty_print()
        else:
            data.pretty_print()
